In [0]:
%pip install numpy pandas scikit-learn imbalanced-learn tensorflow mlflow matplotlib --quiet
#packages
import numpy as np
import pandas as pd
import json

from pyspark.sql.functions import percentile_approx, when

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold, ParameterSampler
from sklearn.metrics import *
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import RandomOverSampler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping

import mlflow
import mlflow.tensorflow

import matplotlib.pyplot as plt


In [0]:
%restart_python

In [0]:
df = spark.table("workspace.default.cb_1_cb_2_features_v_2").dropna()

percentiles = df.agg(
    percentile_approx("cb2_p", 0.33).alias("p33"),
    percentile_approx("cb2_p", 0.66).alias("p66")
).first()

p33, p66 = percentiles["p33"], percentiles["p66"]

df = df.withColumn(
    "cb2_affinity_category",
    when(df.cb2_p <= p33, "weak")
    .when(df.cb2_p <= p66, "moderate")
    .otherwise("strong")
)

pdf = df.toPandas()

X_morgan = np.vstack(pdf['morgan_fp_str'].apply(lambda x: np.array(json.loads(x), dtype=np.float32)))
X_maccs  = np.vstack(pdf['maccs_fp_str'].apply(lambda x: np.array(json.loads(x), dtype=np.float32)))
X_rdkit  = np.vstack(pdf['rdkit_desc50_str'].apply(lambda x: np.array(json.loads(x), dtype=np.float32)))

X_combined = np.hstack([X_morgan, X_maccs, X_rdkit])

le = LabelEncoder()
y_clf = le.fit_transform(pdf['cb2_affinity_category'])
y_reg = pdf['cb2_p'].values

feature_sets = {
    "morgan": X_morgan,
    "maccs": X_maccs,
    "rdkit": X_rdkit,
    "combined": X_combined
}

In [0]:
%restart_python

In [0]:
display(
    df.groupBy("cb2_affinity_category")
      .count()
      .filter(df.cb2_affinity_category.isin(["moderate", "strong", "weak"]))
)

In [0]:
def build_clf_model(input_dim, layers, dropout, lr, activation):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(input_dim,)))
    for u in layers:
        model.add(keras.layers.Dense(u, activation=activation))
        model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(3, activation='softmax'))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def build_reg_model(input_dim, layers, dropout, lr, activation):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(input_dim,)))
    for u in layers:
        model.add(keras.layers.Dense(u, activation=activation))
        model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='mse',
                  metrics=['mae'])
    return model

In [0]:
param_grid = {
    "layers": [[128, 64], [256, 128]],
    "dropout": [0.2, 0.4],
    "lr": [1e-3, 1e-4],
    "batch_size": [32, 64],
    "epochs": [100],
    "activation": ["relu", "tanh"]
}

random_params = list(ParameterSampler(param_grid, n_iter=8, random_state=42))
kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

clf_results = []

for fname, X in feature_sets.items():
    for params in random_params:
        f1s, accs, precs, recs, aucs = [], [], [], [], []

        for train_idx, val_idx in kf.split(X, y_clf):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y_clf[train_idx], y_clf[val_idx]

            if fname in ["rdkit", "combined"]:
                scaler = StandardScaler()
                X_tr = scaler.fit_transform(X_tr)
                X_val = scaler.transform(X_val)

            ros = RandomOverSampler(random_state=42)
            X_tr, y_tr = ros.fit_resample(X_tr, y_tr)

            class_weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
            class_weight_dict = dict(enumerate(class_weights))

            model = build_clf_model(X.shape[1], params["layers"], params["dropout"], params["lr"], params["activation"])

            early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

            history = model.fit(X_tr, y_tr,
                                validation_data=(X_val, y_val),
                                epochs=params["epochs"],
                                batch_size=params["batch_size"],
                                class_weight=class_weight_dict,
                                callbacks=[early_stop],
                                verbose=0)

            pred_prob = model.predict(X_val, verbose=0)
            pred = np.argmax(pred_prob, axis=1)

            accs.append(accuracy_score(y_val, pred))
            precs.append(precision_score(y_val, pred, average='weighted', zero_division=0))
            recs.append(recall_score(y_val, pred, average='weighted', zero_division=0))
            f1s.append(f1_score(y_val, pred, average='weighted', zero_division=0))

            try:
                aucs.append(roc_auc_score(y_val, pred_prob, multi_class='ovr'))
            except:
                aucs.append(np.nan)

        with mlflow.start_run():
            mlflow.log_params(params)
            mlflow.log_param("feature", fname)
            mlflow.log_metric("f1", np.mean(f1s))

        clf_results.append({
            "feature": fname,
            "params": params,
            "F1_mean": np.mean(f1s),
            "F1_std": np.std(f1s),
            "Accuracy_mean": np.mean(accs),
            "Accuracy_std": np.std(accs),
            "Precision_mean": np.mean(precs),
            "Recall_mean": np.mean(recs),
            "ROC_AUC_mean": np.nanmean(aucs)
        })

clf_df = pd.DataFrame(clf_results)
best_clf = clf_df.loc[clf_df.groupby("feature")["F1_mean"].idxmax()]

display(clf_df.sort_values("F1_mean", ascending=False))
display(best_clf)

cm = confusion_matrix(y_val, pred)
plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.show()

plt.figure()
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Loss Curve")
plt.show()

In [0]:
param_grid = {
    "layers": [[128, 64], [256, 128]],
    "dropout": [0.2, 0.4],
    "lr": [1e-3, 1e-4],
    "batch_size": [32, 64],
    "epochs": [100],
    "activation": ["relu", "tanh"]
}

random_params = list(ParameterSampler(param_grid, n_iter=8, random_state=42))
kf = KFold(n_splits=3, shuffle=True, random_state=42)

reg_results = []

for fname, X in feature_sets.items():
    for params in random_params:
        rmses, maes, r2s = [], [], []

        for train_idx, val_idx in kf.split(X):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y_reg[train_idx], y_reg[val_idx]

            if fname in ["rdkit", "combined"]:
                scaler = StandardScaler()
                X_tr = scaler.fit_transform(X_tr)
                X_val = scaler.transform(X_val)

            model = build_reg_model(X.shape[1], params["layers"], params["dropout"], params["lr"], params["activation"])

            early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

            history = model.fit(X_tr, y_tr,
                                validation_data=(X_val, y_val),
                                epochs=params["epochs"],
                                batch_size=params["batch_size"],
                                callbacks=[early_stop],
                                verbose=0)

            pred = model.predict(X_val, verbose=0).flatten()

            rmses.append(np.sqrt(mean_squared_error(y_val, pred)))
            maes.append(mean_absolute_error(y_val, pred))
            r2s.append(r2_score(y_val, pred))

        with mlflow.start_run():
            mlflow.log_params(params)
            mlflow.log_param("feature", fname)
            mlflow.log_metric("rmse", np.mean(rmses))

        reg_results.append({
            "feature": fname,
            "params": params,
            "RMSE_mean": np.mean(rmses),
            "RMSE_std": np.std(rmses),
            "MAE_mean": np.mean(maes),
            "R2_mean": np.mean(r2s)
        })

reg_df = pd.DataFrame(reg_results)
best_reg = reg_df.loc[reg_df.groupby("feature")["RMSE_mean"].idxmin()]

display(reg_df.sort_values("RMSE_mean"))
display(best_reg)

plt.figure()
plt.scatter(y_val, pred)
plt.title("Predicted vs Actual")
plt.show()

plt.figure()
plt.scatter(pred, y_val - pred)
plt.title("Residual Plot")
plt.show()

plt.figure()
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Loss Curve")
plt.show()

In [0]:
clf_best = best_clf[["feature", "F1_mean", "Accuracy_mean", "Precision_mean", "Recall_mean", "ROC_AUC_mean"]]
reg_best = best_reg[["feature", "RMSE_mean", "MAE_mean", "R2_mean"]]

final_table = clf_best.merge(reg_best, on="feature", how="inner")

display(final_table)
